# 07 — RAG Document Intelligence\n\nEnd-to-end local RAG pipeline over the platform's policy documents (`data/documents/*.md`): parse with Docling-style layout-aware chunking (`rag/local_stack/document_parser.py`), embed with a local multilingual `sentence-transformers` model (`rag/embeddings/embedder.py`), index into ChromaDB (`rag/local_stack/vector_store.py`), then retrieve for real queries and score retrieval quality against a golden Q&A set (`rag/evaluation/`).\n\nThis is the **local-first equivalent of Snowflake Cortex Search** (ADR-011, `docs/decisions/ADR-011-*.md`) — same retrieval contract (embed query → similarity search → top-k chunks + scores), but running entirely offline against a local ChromaDB collection instead of a live Snowflake Cortex service. Nothing in this notebook calls an LLM; it demonstrates retrieval only, which is exactly the part of RAG that doesn't need one.\n\nAll modules imported below are the real production modules under `rag/` — nothing here is a notebook-local reimplementation."

In [1]:
import os
import sys
from pathlib import Path

# This environment has no reliable internet/DNS access. sentence-transformers' model is already
# cached locally, but without HF_HUB_OFFLINE the huggingface_hub client still tries an online
# freshness check first and raises when it can't reach the network. Force offline mode so the
# cached model loads directly, no network call attempted.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DOCS_DIR = ROOT / "data" / "documents"
required_docs = ["refund_policy.md", "delivery_policy.md", "privacy_policy.md", "loyalty_policy.md"]
missing = [d for d in required_docs if not (DOCS_DIR / d).exists()]
if missing:
    raise FileNotFoundError(f"Missing policy documents in {DOCS_DIR}: {missing}")

print(f"ROOT resolved to: {ROOT}")
print(f"Policy documents found: {[p.name for p in sorted(DOCS_DIR.glob('*.md'))]}")

ROOT resolved to: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\Enterprise Customer Intelligence Data Platform (Terminar)
Policy documents found: ['delivery_policy.md', 'loyalty_policy.md', 'privacy_policy.md', 'refund_policy.md']


## 1. Ingest the policy documents (real pipeline)\n\nCalls `rag.embeddings.ingest.ingest_documents()` unmodified — the same orchestration function `rag/README.md` documents as the production ingestion entrypoint. Under the hood it:\n\n1. Calls `rag.parsing.load_and_parse_documents()`, which runs `DoclingDocumentParser` (`rag/local_stack/document_parser.py`) over each Markdown file, producing layout-aware chunks (`DocumentChunk`, tagged with `doc_type`/`section`/`kind` metadata).\n2. Embeds every chunk with `LocalEmbedder` (`rag/embeddings/embedder.py`) — a local `sentence-transformers` multilingual model (384-dim), the offline stand-in for Snowflake `AI_EMBED`.\n3. Upserts the embedded chunks into `ChromaVectorStore` (`rag/local_stack/vector_store.py`), persisted at `data/rag/chroma/` — the same collection the FastAPI layer and agents would query in production.\n\nThis is a real, first-class model load (multilingual MiniLM) — the first call below may take a few seconds while the model downloads/loads from the local cache.

In [2]:
import time

from rag.embeddings.embedder import LocalEmbedder
from rag.embeddings.ingest import ingest_documents

t0 = time.time()
embedder = LocalEmbedder()
vector_store, chunk_count = ingest_documents(documents_dir=DOCS_DIR, embedder=embedder)
elapsed = time.time() - t0

print(f"Embedding model: {embedder.model_name}")
print(f"Ingested {chunk_count} chunks from {len(required_docs)} documents in {elapsed:.1f}s")
print(f"Persisted to: {vector_store.persist_directory}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model: paraphrase-multilingual-MiniLM-L12-v2
Ingested 15 chunks from 4 documents in 53.5s
Persisted to: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\Enterprise Customer Intelligence Data Platform (Terminar)\data\rag\chroma


## 2. Real retrieval queries\n\nA handful of natural-language questions, embedded with the same `LocalEmbedder`, queried against the live `ChromaVectorStore` via `store.query()`. Each result shows the retrieved chunk's `doc_type`, its similarity score (Chroma's squared-L2 distance converted to `1/(1+distance)`, per `vector_store.py`), and a text preview — the exact contract `mcp/tools` and any downstream RAG-consuming agent would receive.

In [3]:
demo_queries = [
    "How many days does a customer have to request a refund after delivery?",
    "What happens if a delivery fails multiple times?",
    "How does LGPD affect what customer data we can keep?",
    "How does a customer reach VIP loyalty status?",
]

for query in demo_queries:
    query_embedding = embedder([query])[0]
    matches = vector_store.query(query_embedding, top_k=3)
    print(f"Q: {query}")
    for rank, m in enumerate(matches, start=1):
        preview = m.record.text[:110].replace("\n", " ")
        print(f"   [{rank}] score={m.score:.4f}  doc_type={m.record.metadata.get('doc_type')!r}  chunk_id={m.record.id!r}")
        print(f"        \"{preview}...\"")
    print()

Q: How many days does a customer have to request a refund after delivery?
   [1] score=0.7101  doc_type='refund_policy'  chunk_id='refund_policy-3'
        "Process: Customer opens a support ticket with category refund_request . Support validates order status and del..."
   [2] score=0.6876  doc_type='refund_policy'  chunk_id='refund_policy-1'
        "Eligibility: Customers may request a refund within 30 days of the delivery date if: The product arrived damage..."
   [3] score=0.5734  doc_type='delivery_policy'  chunk_id='delivery_policy-3'
        "Failed delivery attempts: After 3 failed delivery attempts, the order is returned to the seller and the custom..."

Q: What happens if a delivery fails multiple times?
   [1] score=0.5674  doc_type='delivery_policy'  chunk_id='delivery_policy-3'
        "Failed delivery attempts: After 3 failed delivery attempts, the order is returned to the seller and the custom..."
   [2] score=0.4742  doc_type='delivery_policy'  chunk_id='delivery_polic

## 3. Retrieval quality — precision@3 on the golden Q&A set\n\n`rag/evaluation/precision_at_k.py::evaluate_precision_at_k()` re-ingests the four documents into a fresh, isolated Chroma collection and scores retrieval against `rag/evaluation/golden_questions.py`'s 15 hand-written questions (each grounded in the actual document text, per Constitution Article IV — no invented facts). A question counts as a "hit" at k=3 if any of the top-3 retrieved chunks' `doc_type` matches the question's expected document(s).\n\nRun live below (takes a similar amount of time to the ingestion step, since it re-embeds everything in an isolated store for a repeatable score), then cross-checked against the already-committed `rag/evaluation/precision_at_k_results.json` to confirm the numbers agree.

In [4]:
from rag.evaluation.precision_at_k import evaluate_precision_at_k

t0 = time.time()
report = evaluate_precision_at_k(k=3)
elapsed = time.time() - t0

print(f"Indexed {report['chunk_count']} chunks ({elapsed:.1f}s)")
print(
    f"precision@{report['k']} = {report['precision_at_k']} "
    f"({report['hits']}/{report['num_questions']}) "
    f"target={report['target']} meets_target={report['meets_target']}"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Indexed 15 chunks (4.5s)
precision@3 = 1.0 (15/15) target=0.8 meets_target=True


In [5]:
import json

# Cross-check against the already-committed results file (from a prior real run) — the score
# is deterministic given the same documents/questions/model, so these should agree.
committed_path = ROOT / "rag" / "evaluation" / "precision_at_k_results.json"
committed = json.loads(committed_path.read_text(encoding="utf-8"))
print(f"Committed precision@{committed['k']} = {committed['precision_at_k']}  (this run: {report['precision_at_k']})")
print(f"Match: {committed['precision_at_k'] == report['precision_at_k']}")

print("\nExample Q&A pairs (from this live run):")
for r in report["results"][:3]:
    top = r["retrieved"][0]
    print(f"\nQ: {r['question']}")
    print(f"   expected doc_type(s): {r['expected_doc_types']}  hit={r['hit']}")
    print(f"   top-1 retrieved: doc_type={top['doc_type']!r} score={top['score']}")
    print(f"   \"{top['text']}...\"")

Committed precision@3 = 1.0  (this run: 1.0)
Match: True

Example Q&A pairs (from this live run):

Q: How many days does a customer have to request a refund after delivery?
   expected doc_type(s): ['refund_policy']  hit=True
   top-1 retrieved: doc_type='refund_policy' score=0.7101
   "Process: Customer opens a support ticket with category refund_request . Support validates order status and delivery date..."

Q: What is the right of withdrawal (arrependimento) window under Brazilian consumer law?
   expected doc_type(s): ['refund_policy']  hit=True
   top-1 retrieved: doc_type='refund_policy' score=0.5291
   "Eligibility: Customers may request a refund within 30 days of the delivery date if: The product arrived damaged or mater..."

Q: Which refunds require supervisor approval / human-in-the-loop?
   expected doc_type(s): ['refund_policy']  hit=True
   top-1 retrieved: doc_type='refund_policy' score=0.6093
   "Process: Customer opens a support ticket with category refund_request . Sup

## Maps to the real platform\n\n- `rag/local_stack/` — the local-first vector store abstraction (ChromaDB primary, FAISS documented alternative) that this notebook exercises directly.\n- ADR-011 (`docs/decisions/ADR-011-*.md`) — documents this local stack as the parity implementation of Snowflake Cortex Search: same retrieval contract, swappable backend, no vendor lock-in for local development.\n- `api/` and `agents/` would sit on top of this exact retrieval call (`store.query(embedding, top_k=k)`) to ground an LLM's answer in the policy documents — the part this notebook does **not** demonstrate, since `agents/llm_gateway/router.py`'s provider adapters are stubs in this environment (no API key configured). Retrieval is fully real and fully offline; the generation step on top of it is a documented TODO for whoever adds a live LLM key later.\n- `08_agents_mcp_a2a_demo.ipynb` — picks up where this leaves off, showing the agent/tool orchestration layer with the same "real mechanics, no live LLM" honesty.